# Ya-Risk — V24 SCHEMA-FIRST — Qwen3.8 — Série G n°2

Objectif : extraction rapide et **fermée par schéma** de la liasse fiscale, avec ACTIF / PASSIF / TCR actifs aujourd'hui et référentiel complet A01–A13 prêt pour la suite.

Principes V24 :
- une page est d'abord rattachée à un **gabarit officiel** ; une ligne isolée ne suffit jamais à décider le tableau ;
- Qwen ne peut retourner que des identifiants courts appartenant au schéma de la page ; Python reconstruit les `field_id` uniques ;
- TCR = 4 cases atomiques par ligne : `N.DEBIT`, `N.CREDIT`, `N1.DEBIT`, `N1.CREDIT`, identiques sur les deux pages ;
- Qwen lit, Python range et contrôle ; aucune valeur n'est déplacée Débit↔Crédit ou N↔N-1 ;
- les valeurs imprimées des lignes calculées sont conservées puis comparées aux formules Excel après recalcul LibreOffice ;
- alerte d'écart à partir de **1 000 DZD**, niveaux **LOW / MEDIUM / HIGH** ;
- un sous-dossier = un client ; plusieurs PDF = plusieurs exercices ; un seul Excel multi-exercices est produit.


In [ ]:
# CELLULE 1 — DEPENDANCES LEGERES
# Ne pas réinstaller torch/vLLM : ils proviennent de l'environnement GPU Domino.
%pip install -q -U "transformers>=4.57.0" accelerate pymupdf pillow psutil openpyxl
print("✅ Dépendances Python légères OK")


In [ ]:
# CELLULE 2 — IMPORTS
import os, re, json, time, gc, copy, shutil, subprocess, sys, unicodedata, traceback, tempfile
from pathlib import Path
from datetime import datetime
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import fitz
try:
    fitz.TOOLS.mupdf_display_errors(False)
    fitz.TOOLS.mupdf_display_warnings(False)
except Exception:
    pass
import torch
from PIL import Image
import openpyxl
from openpyxl import load_workbook
from openpyxl.comments import Comment
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side

print("✅ Imports OK")


In [ ]:
# CELLULE 3 — CONFIGURATION V24
MODEL_NAME = os.environ.get("QWEN_MODEL_NAME", "Qwen3.8-27B-FP8")
MODEL_PATH = os.environ.get(
    "QWEN_MODEL_PATH",
    "/domino/edv/modelhub/ModelHub-model-huggingface-Qwen/Qwen3.8-27B-FP8/main"
)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

INPUT_DIR = Path("/mnt/Risk/bilans_in")
OUTPUT_DIR = Path("/mnt/Risk/bilans_out/GT/V24")
JSON_DIR = OUTPUT_DIR / "json_bruts_v24"
CONTROL_DIR = OUTPUT_DIR / "json_avant_certification_v24"
XLSX_DIR = OUTPUT_DIR / "excel_pre_certification_v24"
LOG_PATH = OUTPUT_DIR / "pipeline_v24.log"
INDEX_PATH = OUTPUT_DIR / "index_v24.jsonl"
for p in (OUTPUT_DIR, JSON_DIR, CONTROL_DIR, XLSX_DIR): p.mkdir(parents=True, exist_ok=True)

# Extraction active aujourd'hui. Les annexes sont classées et décrites par le schéma, sans extraction détaillée.
ACTIVE_TABLES = {"ACTIF", "PASSIF", "TCR"}
SCHEMA_VERSION = "24.0"
SCHEMA_REGISTRY_VERSION = "SERIE_G2_2011_SCHEMA_FIRST_V1"
FORCE_REEXTRACT = False

# Profil vitesse / image (V23 avait validé ce principe sur votre H100).
ROUTER_MAX_SIDE = 768
EXTRACT_MAX_SIDE = 1536
RETRY_MAX_SIDE = 2024
PDF_ROUTER_ZOOM = 1.15
PDF_EXTRACT_ZOOM = 2.25
BLANK_THRESHOLD = 0.985
CLASSIF_BATCH_SIZE = 16
GPU_BATCH_SIZE = 8
PDF_WORKERS = 4

# Retry uniquement si le résultat est anormalement pauvre — ce seuil ne décide jamais du type de page.
MIN_VALUES_RETRY = {"ACTIF":10, "PASSIF":8, "TCR_P1":8, "TCR_P2":4}

# Politique d'écarts demandée — DZD absolus.
ALERT_MIN_DZD = 1_000.0
ALERT_MEDIUM_DZD = 10_000.0
ALERT_HIGH_DZD = 100_000.0

def ecart_level(ecart_dzd):
    a = abs(float(ecart_dzd or 0.0))
    if a < ALERT_MIN_DZD: return "OK"
    if a < ALERT_MEDIUM_DZD: return "LOW"
    if a < ALERT_HIGH_DZD: return "MEDIUM"
    return "HIGH"

if DEVICE == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision("high")

print("Modele:", MODEL_NAME)
print("Device:", DEVICE)
print("PDF détectés:", len(list(INPUT_DIR.rglob("*.pdf"))) if INPUT_DIR.exists() else 0)
print("Seuils écarts DZD:", ALERT_MIN_DZD, ALERT_MEDIUM_DZD, ALERT_HIGH_DZD)


In [ ]:
# CELLULE 4 — LOG + OUTILS GENERAUX
def log(msg):
    line = datetime.now().strftime("%Y-%m-%d %H:%M:%S") + " — " + str(msg)
    print(line)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    with open(LOG_PATH, "a", encoding="utf-8") as f: f.write(line + "\n")

def strip_accents(s):
    return "".join(c for c in unicodedata.normalize("NFKD", str(s or "")) if not unicodedata.combining(c))

def norm_text(s):
    return re.sub(r"[^a-z0-9]+", " ", strip_accents(s).lower()).strip()

def norm_nif(v):
    d = re.sub(r"\D", "", str(v or ""))
    return d if len(d) == 15 else None

def norm_amount(v):
    if v is None or isinstance(v, bool): return None
    if isinstance(v, (int,float)): return float(v)
    s = str(v).strip().replace(" ", " ")
    if not s or s.lower() in {"null","none","-","—",""}: return None
    neg = s.startswith("(") and s.endswith(")")
    s = s.strip("()").replace(" ", "").replace("DZD", "").replace("DA", "")
    # Les liasses algériennes utilisent fréquemment l'espace comme séparateur de milliers.
    # Si une virgule est le seul séparateur décimal, elle est convertie en point.
    if "," in s and "." not in s and len(s.rsplit(",",1)[-1]) <= 2:
        s = s.replace(",", ".")
    else:
        s = s.replace(",", "")
    s = re.sub(r"[^0-9.\-+]", "", s)
    try:
        x = float(s)
        return -x if neg else x
    except Exception:
        return None

def parse_year(v):
    if v is None: return None
    s = str(v)
    ys = re.findall(r"(?<!\d)(20\d{2}|19\d{2})(?!\d)", s)
    if ys: return int(ys[-1])
    # dates abrégées jj/mm/23
    m = re.search(r"(?:^|\D)(\d{2})\s*$", s)
    if m:
        yy=int(m.group(1)); return 2000+yy if yy < 70 else 1900+yy
    return None

def chunks(seq, n):
    for i in range(0, len(seq), n): yield seq[i:i+n]


In [ ]:
# CELLULE 5 — REFERENTIEL OFFICIEL SERIE G N°2 + FIELD_ID UNIQUES
TABLE_SCHEMAS = {'ACTIF': {'titre': 'BILAN (ACTIF)',
           'cols': ['montant_brut', 'amortissements_provisions_pertes', 'net_n', 'net_n1'],
           'postes': {'ecarts_acquisition_goodwill': 'Ecart d acquisition goodwill',
                      'immobilisations_incorporelles': 'Immobilisations incorporelles',
                      'terrains': 'Terrains',
                      'batiments': 'Batiments',
                      'autres_immobilisations_corporelles': 'Autres Immobilisations corporelles',
                      'immobilisations_corporelles': 'Immobilisations corporelles',
                      'immobilisations_en_concession': 'Immobilisations en concession',
                      'immobilisations_en_cours': 'Immobilisations en cours',
                      'titres_mis_en_equivalence': 'Titres mis en equivalence',
                      'autres_participations_creances': 'Autres participations et creances rattachees',
                      'autres_titres_immobilises': 'Autres titres immobilises',
                      'prets_actifs_financiers_non_courants': 'Prets et autres actifs financiers non courants',
                      'impots_differes_actif': 'Impots Differes Actif',
                      'immobilisations_financieres': 'Immobilisations financieres',
                      'total_actif_non_courant': 'TOTAL ACTIF NON COURANT',
                      'stocks_encours': 'Stocks et encours',
                      'clients': 'Clients',
                      'autres_debiteurs': 'Autres debiteurs',
                      'impots_assimiles_actif': 'Impots et assimiles',
                      'autres_creances_assimiles': 'Autres Creances et Emplois assimiles',
                      'creances_emplois_assimiles': 'Creances et emplois assimiles',
                      'disponibilites_assimiles': 'Disponibilites et assimiles',
                      'placements_financiers_courants': 'Placements et autres actifs financiers courants',
                      'tresorerie_actif': 'Tresorerie',
                      'total_actif_courant': 'TOTAL ACTIF COURANT',
                      'total_general_actif': 'TOTAL GENERAL ACTIF'},
           'table_id': 'ACTIF'},
 'PASSIF': {'titre': 'BILAN (PASSIF)',
            'cols': ['n', 'n1'],
            'postes': {'capital_emis': 'Capital emis',
                       'capital_non_appele': 'Capital non appele',
                       'primes_reserves': 'Primes et reserves',
                       'ecart_reevaluation': 'Ecart de reevaluation',
                       'ecart_equivalence': 'Ecart d equivalence',
                       'resultat_net_passif': 'Resultat net',
                       'report_a_nouveau': 'Report a nouveau',
                       'part_societe_consolidante': 'Part de la societe consolidante',
                       'part_minoritaires': 'Part des minoritaires',
                       'total_capitaux_propres': 'TOTAL I',
                       'emprunts_dettes_financieres': 'Emprunts et dettes financieres',
                       'impots_differes_provisionnes': 'Impots differes et provisionnes',
                       'autres_dettes_non_courantes': 'Autres dettes non courantes',
                       'provisions_produits_avance': 'Provisions et produits constatés d avance',
                       'total_passifs_non_courants': 'TOTAL PASSIFS NON COURANTS II',
                       'fournisseurs_rattaches': 'Fournisseurs et comptes rattaches',
                       'impots_passif': 'Impots',
                       'autres_dettes': 'Autres dettes',
                       'tresorerie_passif': 'Tresorerie Passif',
                       'total_passifs_courants': 'TOTAL PASSIFS COURANTS',
                       'total_general_passif': 'TOTAL GENERAL PASSIF'},
            'table_id': 'PASSIF'},
 'TCR': {'titre': 'COMPTE DE RESULTAT',
         'cols': ['n_debit', 'n_credit', 'n1_debit', 'n1_credit'],
         'postes': {'ventes_marchandises': 'Ventes de Marchandises',
                    'produits_fabriques': 'Produits Fabriques',
                    'prestations_services': 'Prestations de Services',
                    'ventes_travaux': 'Ventes de Travaux',
                    'produits_annexes': 'Produits Annexes',
                    'rabais_remises_ristournes_accordes': 'Rabais remises ristournes accordes',
                    'chiffre_affaires_net': 'Chiffre d affaires net',
                    'production_stockee_destockee': 'Production Stockee ou destockee',
                    'production_immobilisee': 'Production immobilisee',
                    'subvention_exploitation': 'Subvention d exploitation',
                    'production_exercice': 'I-Production de l exercice',
                    'achats_marchandises_vendues': 'Achats de Marchandises vendues',
                    'matieres_premieres': 'Matieres premieres',
                    'autres_approvisionnements': 'Autres Approvisionnements',
                    'variation_stocks': 'Variation des Stocks',
                    'achats_etudes_prestations': 'Achats d Etudes et de Prestations de services',
                    'rabais_remises_ristournes_obtenus_achats': 'Rabais remises ristournes obtenus sur achats',
                    'autres_consommations': 'Autres consommations',
                    'sous_traitance_generale': 'Sous-traitance generale',
                    'locations': 'Locations',
                    'entretien_reparations': 'Entretien reparations et maintenance',
                    'primes_assurances': 'Primes d assurances',
                    'personnel_exterieur': 'Personnel exterieur a l entreprise',
                    'remuneration_intermediaires': 'Remuneration d intermediaires et honoraires',
                    'publicite': 'Publicite',
                    'deplacements_missions': 'Deplacements missions et receptions',
                    'rabais_remises_ristournes_obtenus_services': 'Rabais remises ristournes obtenus sur services exterieurs',
                    'autres_services': 'Autres services',
                    'consommations_exercice': 'II-Consommations de l exercice',
                    'valeur_ajoutee_exploitation': 'III-Valeur ajoutee d exploitation',
                    'charges_personnel': 'Charges de personnel',
                    'impots_taxes_assimiles': 'Impots et taxes et versements assimiles',
                    'excedent_brut_exploitation': 'IV-Excedent brut d exploitation',
                    'autres_produits_operationnels': 'Autres produits operationnels',
                    'autres_charges_operationnelles': 'Autres charges operationnelles',
                    'dotations_amortissements': 'Dotations aux amortissements',
                    'provisions': 'Provisions',
                    'pertes_valeur': 'Perte de Valeur',
                    'reprises_pertes_valeur_provisions': 'Reprise sur pertes de valeur et provisions',
                    'resultat_operationnel': 'V-Resultat operationnel',
                    'produits_financiers': 'Produits financiers',
                    'charges_financieres': 'Charges financieres',
                    'resultat_financier': 'VI-Resultat Financier',
                    'resultat_ordinaire': 'VII-Resultat ordinaire',
                    'elements_extraordinaires_produits': 'Elements extraordinaires Produits',
                    'elements_extraordinaires_charges': 'Elements extraordinaires Charges',
                    'resultat_extraordinaire': 'VIII-Resultat extraordinaire',
                    'impots_exigibles_resultats': 'Impots exigibles sur resultats',
                    'impots_differes_resultats': 'Impots differes sur resultats',
                    'resultat_net_exercice': 'RESULTAT NET DE L EXERCICE'},
         'table_id': 'TCR'},
 'A01': {'titre': '1/ Tableau des mouvements des stocks',
         'cols': ['solde_debut', 'debit', 'credit', 'solde_fin'],
         'postes': {'stocks_marchandises': 'Stocks de marchandises',
                    'matieres_fournitures': 'Matieres et fournitures',
                    'autres_approvisionnements': 'Autres approvisionnements',
                    'encours_production_biens': 'Encours de production de biens',
                    'encours_production_services': 'Encours de production de services',
                    'stocks_produits': 'Stocks de produits',
                    'stocks_provenant_immobilisations': 'Stocks provenant d immobilisations',
                    'stocks_exterieur': 'Stocks a l exterieur',
                    'total': 'TOTAL'},
         'table_id': 'A01'},
 'A03': {'titre': '3/ Charges de personnel, impots, taxes, autres services',
         'cols': ['montant'],
         'postes': {'charges_locatives': 'Charges locatives et charges de copropriete',
                    'etudes_recherches': 'Etudes et recherches',
                    'documentation_divers': 'Documentation et divers',
                    'transports_biens': 'Transports de biens et transport collectif du personnel',
                    'frais_postaux': 'Frais postaux et de telecommunications',
                    'services_bancaires': 'Services bancaires et assimiles',
                    'cotisations_divers': 'Cotisations et divers',
                    'total_autres_services': 'TOTAL (1)',
                    'remunerations_personnel': 'Remunerations du personnel',
                    'remuneration_exploitant': 'Remunerations de l exploitant individuel (cas d une EURL)',
                    'cotisations_sociales': 'Cotisations aux organismes sociaux',
                    'charges_sociales_exploitant': 'Charges sociales de l exploitant individuel (cas d une EURL)',
                    'autres_charges_sociales': 'Autres charges sociales',
                    'autres_charges_personnel': 'Autres charges de personnels',
                    'total_charges_personnel': 'TOTAL (2)',
                    'impots_sur_remunerations': 'Impots taxes et versements assimiles sur remunerations',
                    'impots_non_recuperables': 'Impots et taxes non recuperables sur chiffres d affaires',
                    'autres_impots_taxes': 'Autres impots et taxes (hors impots sur les resultats)',
                    'total_impots': 'TOTAL (3)',
                    'total_general': 'TOTAL (1)+(2)+(3)'},
         'table_id': 'A03'},
 'A04': {'titre': '4/ Autres charges et produits operationnels',
         'cols': ['montant'],
         'postes': {'redevances_concessions_charges': 'Redevances pour concessions brevets licences logiciels (charges)',
                    'moins_values_sorties_actifs': 'Moins values sur sorties d actifs immobilises non financiers',
                    'jetons_presence_charges': 'Jetons de presence (charges)',
                    'pertes_creances_irrecouvrables': 'Perte sur creances irrecouvrables',
                    'quote_part_operations_commun_charges': 'Quote-part de resultat sur operations faites en commun (charges)',
                    'amendes_penalites_dons': 'Amendes et penalites subventions accordees dons et liberalites',
                    'charges_exceptionnelles_gestion': 'Charges exceptionnelles de gestion courante',
                    'autres_charges_gestion': 'Autres charges de gestion courante',
                    'total_charges': 'TOTAL (charges)',
                    'redevances_concessions_produits': 'Redevances pour concessions brevets licences logiciels (produits)',
                    'plus_values_sorties_actifs': 'Plus values sur sorties d actifs immobilises non financiers',
                    'jetons_presence_produits': 'Jetons de presence et remunerations d administrateurs ou de gerant',
                    'quotes_parts_subventions_virees': 'Quotes-parts de subventions d investissement virees au resultat',
                    'quote_part_operations_commun_produits': 'Quote-part de resultat sur operations faites en commun (produits)',
                    'rentrees_creances_amorties': 'Rentree sur creances amorties',
                    'produits_exceptionnels_gestion': 'Produits exceptionnels sur operations de gestion',
                    'autres_produits_gestion': 'Autres produits de gestion courante',
                    'total_produits': 'TOTAL (produits)'},
         'table_id': 'A04'},
 'A05': {'titre': '5/ Tableau des amortissements et pertes de valeurs',
         'cols': ['dotations_cumulees_debut',
                  'dotations_exercice',
                  'diminutions_elements_sortis',
                  'dotations_cumulees_fin',
                  'dotations_fiscales_exercice',
                  'ecarts'],
         'postes': {'goodwill': 'Goodwill',
                    'immobilisations_incorporelles': 'Immobilisations incorporelles',
                    'immobilisations_corporelles': 'Immobilisations corporelles',
                    'participations': 'Participations',
                    'autres_actifs_financiers_non_courants': 'Autres actifs financiers non courants',
                    'total': 'TOTAL'},
         'table_id': 'A05'},
 'A06': {'titre': '6/ Tableau des immobilisations creees ou acquises',
         'cols': ['montants_bruts', 'tva_deduite', 'montant_net_a_amortir'],
         'postes': {'goodwill': 'Goodwill',
                    'immobilisations_incorporelles': 'Immobilisations incorporelles',
                    'immobilisations_corporelles': 'Immobilisations corporelles',
                    'participations': 'Participations',
                    'autres_actifs_financiers_non_courants': 'Autres actifs financiers non courants',
                    'total': 'TOTAL'},
         'table_id': 'A06'},
 'A08': {'titre': '8/ Tableau des provisions et pertes de valeurs',
         'cols': ['provisions_cumulees_debut', 'dotations_exercice', 'reprises_exercice', 'provisions_cumulees_fin'],
         'postes': {'pertes_valeur_stocks': 'Pertes de valeurs sur stocks',
                    'pertes_valeur_creances': 'Pertes de valeurs sur creances',
                    'pertes_valeur_actions': 'Pertes de valeurs sur actions et parts sociales',
                    'provisions_pensions': 'Provisions pour pensions et obligations similaires',
                    'provisions_litiges': 'Provisions sur litiges',
                    'autres_provisions_personnel': 'Autres provisions liees au personnel',
                    'provisions_impots': 'Provisions pour impots',
                    'autres_provisions': 'Autres provisions',
                    'total': 'TOTAL'},
         'table_id': 'A08'},
 'A09': {'titre': '9/ Tableau de determination du resultat fiscal',
         'cols': ['montant'],
         'postes': {'resultat_net_benefice': 'I. Resultat net de l exercice Benefice',
                    'resultat_net_perte': 'I. Resultat net de l exercice Perte',
                    'charges_immeubles_non_affectes': 'Charges des immeubles non affectes directement a l exploitation',
                    'quote_part_cadeaux_publicitaires': 'Quote-part des cadeaux publicitaires non deductibles',
                    'quote_part_sponsoring': 'Quote-part du sponsoring et parrainage non deductibles',
                    'frais_reception': 'Frais de reception non deductibles',
                    'cotisations_dons': 'Cotisations et dons non deductibles',
                    'impots_taxes_non_deductibles': 'Impots et taxes non deductibles',
                    'provisions_non_deductibles': 'Provisions non deductibles',
                    'amortissements_non_deductibles': 'Amortissements non deductibles',
                    'quote_part_frais_rd': 'Quote-part des frais de recherche developpement non deductibles',
                    'amortissements_credit_bail_preneur': 'Amortissements non deductibles credit bail (Preneur)',
                    'loyers_hors_produits_financiers_bailleur': 'Loyers hors produits financiers (bailleur)',
                    'ibs_impot_exigible': 'Impots sur les benefices - Impot exigible sur le resultat',
                    'ibs_impot_differe': 'Impots sur les benefices - Impot differe (variation)',
                    'pertes_valeur_non_deductibles': 'Pertes de valeurs non deductibles',
                    'amendes_penalites': 'Amendes et penalites',
                    'autres_reintegrations': 'Autres reintegrations',
                    'total_reintegrations': 'Total des reintegrations',
                    'plus_values_cession_actif_immobilise': 'Plus values sur cession d elements d actif immobilises',
                    'produits_plus_values_actions_bourse': 'Produits et plus values de cession des actions et titres assimiles cotes en bourse',
                    'revenus_distribution_benefices': 'Revenus provenant de la distribution des benefices',
                    'amortissements_credit_bail_bailleur': 'Amortissements credit bail (Bailleur)',
                    'loyers_hors_charges_financieres_preneur': 'Loyers hors charges financieres (Preneur)',
                    'complement_amortissements': 'Complement d amortissements',
                    'autres_deductions': 'Autres deductions',
                    'total_deductions': 'Total des deductions',
                    'total_deficits_a_deduire': 'Total des deficits a deduire',
                    'resultat_fiscal_benefice': 'Resultat fiscal (I+II-III-IV) Benefice',
                    'resultat_fiscal_deficit': 'Resultat fiscal (I+II-III-IV) Deficit',
                    'deficit_anterieur_1': 'Déficit antérieur 1 (année à lire sur le formulaire)',
                    'deficit_anterieur_2': 'Déficit antérieur 2 (année à lire sur le formulaire)',
                    'deficit_anterieur_3': 'Déficit antérieur 3 (année à lire sur le formulaire)',
                    'deficit_anterieur_4': 'Déficit antérieur 4 (année à lire sur le formulaire)'},
         'table_id': 'A09'},
 'A10': {'titre': '10/ Tableau d affectation du resultat et des reserves (N-1)',
         'cols': ['montant'],
         'postes': {'origine_report_a_nouveau_n1': 'Report a nouveau de l exercice N-1',
                    'origine_resultat_n1': 'Resultat de l exercice N-1',
                    'origine_prelevements_reserves': 'Prelevements sur reserves',
                    'origine_total': 'TOTAL (origine)',
                    'affectation_reserves': 'Reserves',
                    'affectation_augmentation_capital': 'Augmentation du capital',
                    'affectation_dividendes': 'Dividendes',
                    'affectation_report_a_nouveau': 'Report a nouveau',
                    'affectation_total': 'TOTAL (affectation)'},
         'table_id': 'A10'},
 'A02': {'table_id': 'A02',
         'titre': '2/ Tableau de la fluctuation de la production stockée',
         'dynamic': True,
         'cols': ['debit', 'credit', 'solde_debiteur', 'solde_crediteur'],
         'str_cols': [],
         'postes': {}},
 'A07': {'table_id': 'A07',
         'titre': '7/ Tableau des immobilisations cédées (plus ou moins value) au cours de l’exercice',
         'dynamic': True,
         'cols': ['nature_immobilisation_cedee',
                  'date_acquisition',
                  'montant_net_actif',
                  'amortissements_pratiques',
                  'valeur_nette_comptable',
                  'prix_cession',
                  'plus_value',
                  'moins_value'],
         'str_cols': ['nature_immobilisation_cedee', 'date_acquisition'],
         'postes': {}},
 'A08_1': {'table_id': 'A08_1',
           'titre': '8/1 Relevé des pertes de valeurs sur créances',
           'dynamic': True,
           'cols': ['designation_debiteur', 'valeur_creance', 'perte_valeur_constituee'],
           'str_cols': ['designation_debiteur'],
           'postes': {}},
 'A08_2': {'table_id': 'A08_2',
           'titre': '8/2 Relevé des pertes de valeurs sur actions et parts sociales',
           'dynamic': True,
           'cols': ['filiale', 'valeur_nominale_debut', 'perte_valeur_constituee', 'valeur_nette_comptable'],
           'str_cols': ['filiale'],
           'postes': {}},
 'A11': {'table_id': 'A11',
         'titre': '11/ Tableau des participations (filiales et entités associées)',
         'dynamic': True,
         'cols': ['filiale_entite_associee',
                  'capitaux_propres',
                  'dont_capital',
                  'quote_part_capital_pct',
                  'resultat_dernier_exercice',
                  'prets_avances',
                  'dividendes_encaisses',
                  'valeur_comptable_titres'],
         'str_cols': ['filiale_entite_associee'],
         'postes': {}},
 'A12': {'table_id': 'A12',
         'titre': '12/ Commissions et courtages, redevances, honoraires, sous-traitance, rémunérations diverses et frais de siège',
         'dynamic': True,
         'cols': ['designation_beneficiaire', 'nif', 'adresse', 'montant_percu'],
         'str_cols': ['designation_beneficiaire', 'nif', 'adresse'],
         'postes': {}},
 'A13': {'table_id': 'A13',
         'titre': '13/ Taxe sur l’activité professionnelle',
         'dynamic': True,
         'cols': ['lieu_paiement_tap', 'ca_imposable', 'ca_exonere', 'tap_acquittee'],
         'str_cols': ['lieu_paiement_tap'],
         'postes': {}}}

# Le tableau A09 contient aussi jusqu'à 4 lignes de déficits antérieurs avec année variable.
for _i in range(1,5):
    TABLE_SCHEMAS["A09"].get("postes",{}).pop(f"deficit_anterieur_{_i}", None)
TABLE_SCHEMAS["A09"]["dynamic_sections"] = {
    "deficits_anterieurs": {"max_rows":4, "cols":["annee","montant"], "str_cols":["annee"]}
}

# Pages physiques du formulaire officiel : une page peut porter plusieurs tableaux logiques.
PAGE_TEMPLATES = {
    "ACTIF":      {"tables":["ACTIF"], "anchors":["bilan actif","actifs non courants","actifs courants","total general actif"]},
    "PASSIF":     {"tables":["PASSIF"],"anchors":["bilan passif","capitaux propres","passifs non courants","passifs courants","total passif"]},
    "TCR_P1":     {"tables":["TCR"],   "anchors":["compte de resultat","ventes de marchandises","i production de l exercice","ii consommations de l exercice","iv excedent brut d exploitation"]},
    "TCR_P2":     {"tables":["TCR"],   "anchors":["v resultat operationnel","vi resultat financier","vii resultat ordinaire","ix resultat net de l exercice"]},
    "A01_A02":    {"tables":["A01","A02"],"anchors":["1 tableau des mouvements des stocks","2 tableau de la fluctuation de la production stockee"]},
    "A03_A04":    {"tables":["A03","A04"],"anchors":["3 charges de personnel","4 autres charges et produits operationnels"]},
    "A05_A06":    {"tables":["A05","A06"],"anchors":["5 tableau des amortissements et pertes de valeurs","6 tableau des immobilisations creees ou acquises"]},
    "A07_A08":    {"tables":["A07","A08"],"anchors":["7 tableau des immobilisations cedees","8 tableau des provisions et pertes de valeurs"]},
    "A08_1_A08_2":{"tables":["A08_1","A08_2"],"anchors":["8 1 releve des pertes de valeurs sur creances","8 2 releve des pertes de valeurs sur actions"]},
    "A09":        {"tables":["A09"],"anchors":["9 tableau de determination du resultat fiscal","ii reintegrations","iii deductions","resultat fiscal"]},
    "A10_A11":    {"tables":["A10","A11"],"anchors":["10 tableau d affectation du resultat et des reserves","11 tableau des participations"]},
    "A12_A13":    {"tables":["A12","A13"],"anchors":["12 commissions et courtages","13 taxe sur l activite professionnelle"]},
}

# La page 3 et la page 4 forment un seul TCR logique avec les mêmes quatre colonnes atomiques.
TCR_COLUMNS = ["n_debit", "n_credit", "n1_debit", "n1_credit"]
TCR_P1_ROWS = [
 "ventes_marchandises","produits_fabriques","prestations_services","ventes_travaux","produits_annexes",
 "rabais_remises_ristournes_accordes","chiffre_affaires_net","production_stockee_destockee","production_immobilisee",
 "subvention_exploitation","production_exercice","achats_marchandises_vendues","matieres_premieres","autres_approvisionnements",
 "variation_stocks","achats_etudes_prestations","autres_consommations","rabais_remises_ristournes_obtenus_achats",
 "sous_traitance_generale","locations","entretien_reparations","primes_assurances","personnel_exterieur",
 "remuneration_intermediaires","publicite","deplacements_missions","autres_services",
 "rabais_remises_ristournes_obtenus_services","consommations_exercice","valeur_ajoutee_exploitation",
 "charges_personnel","impots_taxes_assimiles","excedent_brut_exploitation"
]
TCR_P2_ROWS = [
 "autres_produits_operationnels","autres_charges_operationnelles","dotations_amortissements","provisions","pertes_valeur",
 "reprises_pertes_valeur_provisions","resultat_operationnel","produits_financiers","charges_financieres","resultat_financier",
 "resultat_ordinaire","elements_extraordinaires_produits","elements_extraordinaires_charges","resultat_extraordinaire",
 "impots_exigibles_resultats","impots_differes_resultats","resultat_net_exercice"
]
assert set(TCR_P1_ROWS + TCR_P2_ROWS) == set(TABLE_SCHEMAS["TCR"]["postes"])
assert TABLE_SCHEMAS["TCR"]["cols"] == TCR_COLUMNS

# Identifiants uniques du contrat de données. Pour les tables dynamiques, l'index de record sera ajouté à l'exécution.
FIELD_REGISTRY = {}
for table_id, spec in TABLE_SCHEMAS.items():
    if spec.get("dynamic"):
        for col in spec["cols"]:
            fid = f"{table_id}.records[].{col}"
            if fid in FIELD_REGISTRY: raise ValueError("field_id dupliqué: "+fid)
            FIELD_REGISTRY[fid] = {"table_id":table_id,"row_id":"records[]","column_id":col}
    else:
        for row_id in spec.get("postes",{}):
            for col in spec["cols"]:
                fid = f"{table_id}.{row_id}.{col}"
                if fid in FIELD_REGISTRY: raise ValueError("field_id dupliqué: "+fid)
                FIELD_REGISTRY[fid] = {"table_id":table_id,"row_id":row_id,"column_id":col}
    for section_id, section in spec.get("dynamic_sections",{}).items():
        for col in section.get("cols",[]):
            fid=f"{table_id}.{section_id}[].{col}"
            if fid in FIELD_REGISTRY: raise ValueError("field_id dupliqué: "+fid)
            FIELD_REGISTRY[fid]={"table_id":table_id,"row_id":section_id+"[]","column_id":col}

print("✅ Schéma officiel chargé:", len(TABLE_SCHEMAS), "tables |", len(FIELD_REGISTRY), "field_id")
print("✅ TCR verrouillé:", TCR_COLUMNS, "| P1", len(TCR_P1_ROWS), "lignes | P2", len(TCR_P2_ROWS), "lignes")


In [ ]:
# CELLULE 6 — ROUTAGE SCHEMA-FIRST (TEXTE D'ABORD, QWEN SEULEMENT SI NECESSAIRE)
def route_from_text(text):
    t = norm_text(text)
    if not t: return None

    # Annexes d'abord : empêche A09 de devenir TCR et A03/A04 de devenir TCR_P2.
    ordered = ["A01_A02","A03_A04","A05_A06","A07_A08","A08_1_A08_2","A09","A10_A11","A12_A13"]
    for template in ordered:
        anchors = PAGE_TEMPLATES[template]["anchors"]
        hits = sum(a in t for a in anchors)
        if hits >= 1 and (template not in {"A09"} or hits >= 2):
            return template

    if "bilan actif" in t and "total general actif" in t: return "ACTIF"
    if "bilan passif" in t and ("capitaux propres" in t or "total passif" in t): return "PASSIF"
    if "compte de resultat" in t and ("debit" in t and "credit" in t): return "TCR_P1"

    # TCR page 2 : exige plusieurs jalons de résultats, jamais une seule occurrence de résultat net.
    p2 = PAGE_TEMPLATES["TCR_P2"]["anchors"]
    if sum(a in t for a in p2) >= 3 and "reintegrations" not in t and "deductions" not in t:
        return "TCR_P2"
    return None

ROUTE_CHOICES = list(PAGE_TEMPLATES) + ["AUTRE"]
PROMPT_ROUTE = """Tu classes UNE page de la liasse fiscale algérienne Série G n°2.
Réponds UNIQUEMENT en JSON strict: {"type":"..."}.
Types autorisés: ACTIF, PASSIF, TCR_P1, TCR_P2, A01_A02, A03_A04, A05_A06, A07_A08, A08_1_A08_2, A09, A10_A11, A12_A13, AUTRE.
Règle fondamentale: classe la STRUCTURE GLOBALE du formulaire, jamais une ligne isolée.
- ACTIF: BILAN ACTIF avec ACTIFS NON COURANTS + ACTIFS COURANTS + TOTAL GENERAL ACTIF.
- PASSIF: BILAN PASSIF avec CAPITAUX PROPRES + PASSIFS NON-COURANTS + PASSIFS COURANTS + TOTAL PASSIF.
- TCR_P1: COMPTE DE RESULTAT, première partie, quatre colonnes N Débit/N Crédit/N-1 Débit/N-1 Crédit.
- TCR_P2: suite du COMPTE DE RESULTAT, lignes V Résultat opérationnel, VI Résultat financier, VII Résultat ordinaire, VIII Résultat extraordinaire, IX Résultat net. La page peut ne pas répéter l'entête des quatre colonnes.
- A09: tableau 9 Détermination du résultat fiscal avec Réintégrations + Déductions; il reste A09 même s'il contient « Résultat net de l'exercice (Compte de résultat) ».
- A03_A04 peut contenir « Autres produits opérationnels » ou « Autres charges opérationnelles » mais ce n'est PAS le TCR.
En cas de doute: AUTRE."""

# Validation structurelle uniforme : elle s'applique à TOUS les gabarits, pas à un cas PASSIF particulier.
def structure_signature(page_template, rowcodes):
    rows=set(rowcodes or [])
    if page_template=="ACTIF":
        ok = ("total_general_actif" in rows and
              bool(rows & {"immobilisations_incorporelles","terrains","immobilisations_financieres","total_actif_non_courant"}) and
              bool(rows & {"stocks_encours","clients","tresorerie_actif","total_actif_courant"}))
        return ok
    if page_template=="PASSIF":
        ok = ("total_general_passif" in rows and
              bool(rows & {"capital_emis","primes_reserves","resultat_net_passif","total_capitaux_propres"}) and
              bool(rows & {"emprunts_dettes_financieres","fournisseurs_rattaches","autres_dettes","total_passifs_courants"}))
        return ok
    if page_template=="TCR_P1":
        return len(rows & {"chiffre_affaires_net","production_exercice","consommations_exercice","valeur_ajoutee_exploitation","excedent_brut_exploitation"}) >= 2
    if page_template=="TCR_P2":
        return len(rows & {"resultat_operationnel","resultat_financier","resultat_ordinaire","resultat_extraordinaire","resultat_net_exercice"}) >= 2
    return True

print("✅ Routeur schema-first prêt")


In [ ]:
# CELLULE 7 — RENDU PDF + INFERENCE vLLM
def resize_image(img, max_side):
    w,h=img.size
    if max(w,h)<=max_side: return img
    r=max_side/max(w,h)
    return img.resize((max(1,int(w*r)), max(1,int(h*r))), Image.LANCZOS)

def estimate_skew(img):
    small=img.convert("L").copy(); small.thumbnail((420,420))
    def score(a):
        r=np.array(small.rotate(a,expand=True,fillcolor=255))<128
        proj=r.sum(axis=1); return float((proj**2).sum())
    coarse=max(range(-8,9,2),key=score)
    best=max([coarse-1,coarse-.5,coarse,coarse+.5,coarse+1],key=score)
    return best if abs(best)>=1 else 0.0

def deskew(img):
    a=estimate_skew(img)
    return (img.rotate(a,expand=True,fillcolor=(255,255,255),resample=Image.BICUBIC),a) if a else (img,0.0)

def is_blank(img):
    small=img.convert("L").copy(); small.thumbnail((500,500)); arr=np.asarray(small)
    return (arr>245).sum()/arr.size >= BLANK_THRESHOLD

def pdf_manifest(path):
    doc=fitz.open(path); out=[]
    for i,p in enumerate(doc):
        r=p.rect
        out.append({"index":i,"text":p.get_text("text") or "","largeur_pdf":float(r.width),"hauteur_pdf":float(r.height)})
    doc.close(); return out

def render_page(path, idx, zoom, max_side, do_deskew=False):
    doc=fitz.open(path); pix=doc.load_page(idx).get_pixmap(matrix=fitz.Matrix(zoom,zoom),alpha=False); doc.close()
    img=Image.frombytes("RGB",[pix.width,pix.height],pix.samples); angle=0.0
    if do_deskew: img,angle=deskew(img)
    img=resize_image(img,max_side)
    return {"index":idx,"image":img,"largeur_px":img.width,"hauteur_px":img.height,"rotation_estimee":angle}

def render_pages(path, indices, zoom, max_side, do_deskew=False):
    indices=list(indices)
    if not indices: return []
    def one(i): return render_page(path,i,zoom,max_side,do_deskew)
    with ThreadPoolExecutor(max_workers=min(PDF_WORKERS,len(indices))) as ex:
        out=list(ex.map(one,indices))
    return sorted(out,key=lambda x:x["index"])

def parse_json_strictish(text):
    if not text: return {}
    s=text.strip().replace("```json","").replace("```","").strip()
    try: return json.loads(s)
    except Exception: pass
    a,b=s.find("{"),s.rfind("}")
    if a>=0 and b>a:
        try: return json.loads(s[a:b+1])
        except Exception: pass
    return {}

def apply_template(messages):
    try: return processor.apply_chat_template(messages,tokenize=False,add_generation_prompt=True,enable_thinking=False)
    except TypeError: return processor.apply_chat_template(messages,tokenize=False,add_generation_prompt=True)

def build_inputs(prompt,images):
    out=[]
    for img in images:
        msgs=[{"role":"user","content":[{"type":"image","image":img},{"type":"text","text":prompt}]}]
        out.append({"prompt":apply_template(msgs),"multi_modal_data":{"image":img}})
    return out

def ask_batch(prompt,images,sampling):
    if not images: return []
    t0=time.time(); outputs=llm.generate(build_inputs(prompt,images),sampling,use_tqdm=False); elapsed=time.time()-t0
    res=[]
    for o in outputs:
        res.append({"text":o.outputs[0].text,"tokens_in":len(o.prompt_token_ids),"tokens_out":len(o.outputs[0].token_ids),"elapsed":elapsed/max(1,len(images))})
    return res

def ask_single(prompt,img,sampling): return ask_batch(prompt,[img],sampling)[0]


In [ ]:
# CELLULE 8 — CHARGEMENT QWEN / vLLM PERFORMANCE
# IMPORTANT : les variables d'environnement sont fixées AVANT import vLLM.
os.environ["VLLM_USE_DEEP_GEMM"]="0"
os.environ["VLLM_USE_FLASHINFER_SAMPLER"]="0"
os.environ["VLLM_ATTENTION_BACKEND"]="FLASH_ATTN"
os.environ["VLLM_WORKER_MULTIPROC_METHOD"]="spawn"

if not torch.cuda.is_available(): raise RuntimeError("CUDA indisponible")
name=torch.cuda.get_device_name(0); props=torch.cuda.get_device_properties(0); total_gib=props.total_memory/1024**3
free_b,_=torch.cuda.mem_get_info(); print(f"GPU={name} | VRAM={total_gib:.1f} GiB | libre={free_b/1024**3:.1f} GiB")

if total_gib < 50:
    VLLM_GPU_UTIL=.96; VLLM_MAX_MODEL_LEN=5120; VLLM_MAX_NUM_SEQS=4; VLLM_MAX_BATCHED_TOKENS=12288
else:
    VLLM_GPU_UTIL=.92; VLLM_MAX_MODEL_LEN=5120; VLLM_MAX_NUM_SEQS=8; VLLM_MAX_BATCHED_TOKENS=32768

from vllm import LLM, SamplingParams
from transformers import AutoProcessor

t0=time.time()
processor=AutoProcessor.from_pretrained(MODEL_PATH,trust_remote_code=True,min_pixels=4*32*32,max_pixels=2000*32*32)
processor.tokenizer.padding_side="left"
llm=LLM(model=MODEL_PATH,dtype="bfloat16",quantization="fp8",gpu_memory_utilization=VLLM_GPU_UTIL,
        max_model_len=VLLM_MAX_MODEL_LEN,max_num_seqs=VLLM_MAX_NUM_SEQS,max_num_batched_tokens=VLLM_MAX_BATCHED_TOKENS,
        kv_cache_dtype="fp8",trust_remote_code=True,enforce_eager=False,enable_prefix_caching=True,
        limit_mm_per_prompt={"image":1},mm_processor_kwargs={"min_pixels":4*32*32,"max_pixels":2000*32*32})
SAMPLING_ROUTE=SamplingParams(temperature=0.0,max_tokens=48,repetition_penalty=1.0)
SAMPLING_EXTRACT=SamplingParams(temperature=0.0,max_tokens=1600,repetition_penalty=1.0)
print("✅ Qwen chargé en",round(time.time()-t0,1),"s | DeepGEMM OFF | eager=False")


In [ ]:
# CELLULE 9 — PROMPTS FERMES PAR SCHEMA + DECODAGE
# Qwen ne reçoit jamais les field_id finaux : seulement des ids courts appartenant au gabarit courant.
def rows_for_template(page_template):
    if page_template=="ACTIF": return list(TABLE_SCHEMAS["ACTIF"]["postes"])
    if page_template=="PASSIF": return list(TABLE_SCHEMAS["PASSIF"]["postes"])
    if page_template=="TCR_P1": return TCR_P1_ROWS
    if page_template=="TCR_P2": return TCR_P2_ROWS
    return []

def table_for_template(page_template):
    return "TCR" if page_template.startswith("TCR_") else page_template

def compact_contract(page_template):
    table=table_for_template(page_template); spec=TABLE_SCHEMAS[table]; rows=rows_for_template(page_template)
    short_to_row={f"r{i:02d}":rc for i,rc in enumerate(rows,1)}
    row_to_short={v:k for k,v in short_to_row.items()}
    return table,spec,short_to_row,row_to_short

def build_extract_prompt(page_template):
    table,spec,short_to_row,_=compact_contract(page_template)
    cols=spec["cols"]
    lines=[
        f"Tu lis UNIQUEMENT le gabarit officiel {page_template} de la liasse fiscale algérienne Série G n°2.",
        "TRANSCRIPTION VISUELLE UNIQUEMENT: ne calcule rien, ne complète rien, ne déplace aucune valeur.",
        "Une case vide ou illisible = null. N'invente jamais une valeur.",
        "Réponds UNIQUEMENT en JSON strict:",
        '{"entete":{"entreprise":null,"nif":null,"exercice":null,"exercice_du":null,"exercice_au":null},"v":{"r01":[...]}}',
        "Retourne uniquement les ids rXX autorisés ayant au moins une valeur non nulle.",
        "Ordre EXACT des colonnes: ["+", ".join(cols)+"]",
    ]
    if table=="TCR":
        lines += [
            "REGLE CRITIQUE TCR: la liste a TOUJOURS 4 positions dans cet ordre immuable:",
            "1=N DEBIT, 2=N CREDIT, 3=N-1 DEBIT, 4=N-1 CREDIT.",
            "La page TCR_P2 conserve exactement ces 4 colonnes même si l'entête n'est pas répété.",
            "Ne transforme jamais Débit en Crédit; ne transforme jamais N en N-1; si la case est incertaine mets null.",
        ]
    lines.append("IDs autorisés:")
    for sid,rc in short_to_row.items(): lines.append(f"{sid} = {spec['postes'][rc]}")
    return "\n".join(lines)

PROMPTS_EXTRACT={t:build_extract_prompt(t) for t in ("ACTIF","PASSIF","TCR_P1","TCR_P2")}

def decode_compact(page_template, data):
    table,spec,short_to_row,_=compact_contract(page_template); cols=spec["cols"]
    raw=(data or {}).get("v") or {}; out={}; unknown=[]
    if not isinstance(raw,dict): raw={}
    for sid,vals in raw.items():
        if sid not in short_to_row:
            unknown.append(sid); continue
        if isinstance(vals,dict): arr=[vals.get(c) for c in cols]
        elif isinstance(vals,(list,tuple)): arr=list(vals)
        else: continue
        arr=(arr+[None]*len(cols))[:len(cols)]
        normalized={c:norm_amount(arr[i]) for i,c in enumerate(cols)}
        if any(v is not None for v in normalized.values()): out[short_to_row[sid]]=normalized
    ent=(data or {}).get("entete") if isinstance((data or {}).get("entete"),dict) else {}
    return table,out,ent,unknown

def count_values(rows): return sum(v is not None for vals in rows.values() for v in vals.values())

print("✅ Prompts schema-first construits")


In [ ]:
# CELLULE 10 — JSON CANONIQUE / OBSERVATIONS UNIQUES
def page_tables_from_template(template): return PAGE_TEMPLATES.get(template,{}).get("tables",[])

def observations_for_page(table,rows,document,page_no):
    obs=[]
    for row_id,vals in rows.items():
        for col,val in vals.items():
            if val is None: continue
            obs.append({
                "field_id":f"{table}.{row_id}.{col}","table_id":table,"row_id":row_id,"column_id":col,"value":val,"unit":"DZD",
                "source":{"document":document,"page":page_no}
            })
    return obs

def classification_page(manifest_page,template,document,route_source):
    n=manifest_page["index"]+1
    blank=template=="BLANCHE"
    return {
        "page_id":f"p{n:03d}","pdf_page_index":manifest_page["index"],"numero_page_scannee":n,"fichier_source":document,
        "classification":{"page_template":template,"types":page_tables_from_template(template),"route_source":route_source,
                          "page_utile":not blank,"page_blanche":blank,"extraction_active":False},
        "entete_page":{},"donnees":{},"observations":[],"statut_extraction":{"statut":"BLANCHE" if blank else "CLASSEE_NON_EXTRAITE"},
        "tokens_in":0,"tokens_out":0,"temps_s":0.0
    }

def extracted_page(rendered,page_template,table,rows,entete,document,route_source,rep,structure_ok,unknown_ids):
    n=rendered["index"]+1
    return {
        "page_id":f"p{n:03d}","pdf_page_index":rendered["index"],"numero_page_scannee":n,"fichier_source":document,
        "classification":{"page_template":page_template,"types":[table],"route_source":route_source,
                          "page_utile":True,"page_blanche":False,"extraction_active":True,"structure_ok":bool(structure_ok)},
        "image":{"largeur_px":rendered.get("largeur_px"),"hauteur_px":rendered.get("hauteur_px"),"rotation_estimee":rendered.get("rotation_estimee",0.0)},
        "entete_page":entete,"donnees":{table:rows},"observations":observations_for_page(table,rows,document,n),
        "statut_extraction":{"statut":"OK" if structure_ok else "A_REVOIR_STRUCTURE","nb_champs_extraits":count_values(rows),"unknown_ids_ignores":unknown_ids},
        "tokens_in":rep.get("tokens_in",0),"tokens_out":rep.get("tokens_out",0),"temps_s":round(rep.get("elapsed",0.0),2)
    }

def build_identity(pages, filename):
    ents=[]
    for p in pages:
        e=p.get("entete_page") or {}
        if e: ents.append(e)
    def majority(field):
        vals=[str(e.get(field)).strip() for e in ents if e.get(field) not in (None,"")]
        return Counter(vals).most_common(1)[0][0] if vals else None
    entreprise=majority("entreprise"); nif=norm_nif(majority("nif"))
    du=majority("exercice_du"); au=majority("exercice_au"); ex=majority("exercice")
    year=parse_year(au) or parse_year(ex)
    if not year:
        ys=re.findall(r"(?<!\d)(20\d{2}|19\d{2})(?!\d)",filename); year=int(ys[-1]) if ys else None
    return {"entreprise":entreprise,"nif_15":nif,"exercice_du":du,"exercice_au":au,"exercice":ex,"annee":year}

def build_document(pdf_path,pages,timings,tokens):
    ident=build_identity(pages,pdf_path.name)
    return {
        "schema_version":SCHEMA_VERSION,"schema_registry_version":SCHEMA_REGISTRY_VERSION,
        "document":{"fichier_source":pdf_path.name,"chemin_source":str(pdf_path),"modele_extraction":MODEL_NAME,"mode_execution":"V24_SCHEMA_FIRST","timings_s":timings},
        "identite":ident,"pages":pages,"observations":[o for p in pages for o in p.get("observations",[])],
        "tokens":{"input":tokens[0],"output":tokens[1]},"controles":{}
    }


In [ ]:
# CELLULE 11 — PIPELINE EXTRACTION RAPIDE
def client_id_from_pdf(pdf_path):
    rel=pdf_path.relative_to(INPUT_DIR); return rel.parts[0] if len(rel.parts)>1 else "_RACINE_"

def json_path_for_pdf(pdf_path):
    d=JSON_DIR/client_id_from_pdf(pdf_path); d.mkdir(parents=True,exist_ok=True); return d/(pdf_path.stem+".json")

def route_with_qwen(pdf_path,manifest,routes,unresolved):
    t_render=t_route=0.0; ti=to=0
    if not unresolved: return t_render,t_route,ti,to
    t0=time.time(); minis=render_pages(pdf_path,unresolved,PDF_ROUTER_ZOOM,ROUTER_MAX_SIDE,False); t_render+=time.time()-t0
    nonblank=[]
    for q in minis:
        if is_blank(q["image"]): routes[q["index"]] = ("BLANCHE","MINI_BLANK")
        else: nonblank.append(q)
    if nonblank:
        t0=time.time(); reps=[]
        for b in chunks(nonblank,CLASSIF_BATCH_SIZE): reps += ask_batch(PROMPT_ROUTE,[x["image"] for x in b],SAMPLING_ROUTE)
        t_route+=time.time()-t0
        for q,rep in zip(nonblank,reps):
            d=parse_json_strictish(rep["text"]); typ=str(d.get("type") or "AUTRE").upper().strip()
            if typ not in ROUTE_CHOICES: typ="AUTRE"
            routes[q["index"]]=(typ,"MINI_QWEN"); ti+=rep["tokens_in"]; to+=rep["tokens_out"]
    return t_render,t_route,ti,to

pdfs=sorted(INPUT_DIR.rglob("*.pdf")) if INPUT_DIR.exists() else []
a_traiter=[p for p in pdfs if FORCE_REEXTRACT or not json_path_for_pdf(p).exists()]
log(f"V24 extraction — à traiter={len(a_traiter)} | déjà V24={len(pdfs)-len(a_traiter)}")
index_records=[]

for num,pdf_path in enumerate(a_traiter,1):
    prefix=f"[{num:04d}/{len(a_traiter):04d}] "; cid=client_id_from_pdf(pdf_path); tpdf=time.time()
    try:
        tok_in=tok_out=0; t_render=t_route=t_extract=0.0; retries=0
        t0=time.time(); manifest=pdf_manifest(pdf_path); t_manifest=time.time()-t0
        routes={}; unresolved=[]
        for p in manifest:
            typ=route_from_text(p.get("text",""))
            if typ: routes[p["index"]]=(typ,"TEXTE_PDF")
            else: unresolved.append(p["index"])
        dr,dq,di,do=route_with_qwen(pdf_path,manifest,routes,unresolved); t_render+=dr; t_route+=dq; tok_in+=di; tok_out+=do

        target=[i for i,(typ,src) in routes.items() if table_for_template(typ) in ACTIVE_TABLES and typ in PROMPTS_EXTRACT]
        counts=Counter(routes.get(i,("AUTRE","DEFAULT"))[0] for i in range(len(manifest)))
        log(prefix+pdf_path.name+" | route "+str(dict(counts)))

        t0=time.time(); rendered=render_pages(pdf_path,target,PDF_EXTRACT_ZOOM,EXTRACT_MAX_SIDE,True); t_render+=time.time()-t0
        by_idx={p["index"]:p for p in rendered}; extracted={}
        for template in ("ACTIF","PASSIF","TCR_P1","TCR_P2"):
            plist=[by_idx[i] for i in target if i in by_idx and routes[i][0]==template]
            if not plist: continue
            prompt=PROMPTS_EXTRACT[template]
            for batch in chunks(plist,GPU_BATCH_SIZE):
                t0=time.time(); reps=ask_batch(prompt,[p["image"] for p in batch],SAMPLING_EXTRACT); t_extract+=time.time()-t0
                for p,rep in zip(batch,reps):
                    tok_in+=rep["tokens_in"]; tok_out+=rep["tokens_out"]
                    raw=parse_json_strictish(rep["text"]); table,rows,ent,unknown=decode_compact(template,raw); nvals=count_values(rows)
                    if nvals < MIN_VALUES_RETRY[template]:
                        t1=time.time(); hd=render_page(pdf_path,p["index"],3.0,RETRY_MAX_SIDE,True); t_render+=time.time()-t1
                        rep2=ask_single(prompt,hd["image"],SAMPLING_EXTRACT); t_extract+=rep2["elapsed"]; tok_in+=rep2["tokens_in"]; tok_out+=rep2["tokens_out"]; retries+=1
                        raw2=parse_json_strictish(rep2["text"]); table2,rows2,ent2,unknown2=decode_compact(template,raw2)
                        if count_values(rows2)>nvals: p,rep,table,rows,ent,unknown=hd,rep2,table2,rows2,ent2,unknown2
                    sig=structure_signature(template,rows.keys())
                    # Fail closed uniquement si le routage venait de Qwen ET la structure extraite ne confirme pas le gabarit.
                    # Si le titre officiel était présent dans la couche texte, on conserve l'extraction et on marque la signature.
                    if routes[p["index"]][1]=="MINI_QWEN" and not sig:
                        log(prefix+f"page {p['index']+1}: {template} non confirmé -> AUTRE/A_REVOIR")
                        routes[p["index"]] = ("AUTRE","REJET_STRUCTURE_SCHEMA")
                        continue
                    extracted[p["index"]]=extracted_page(p,template,table,rows,ent,pdf_path.name,routes[p["index"]][1],rep,sig,unknown)

        pages=[]
        for mp in manifest:
            idx=mp["index"]
            if idx in extracted: pages.append(extracted[idx])
            else:
                typ,src=routes.get(idx,("AUTRE","DEFAULT")); pages.append(classification_page(mp,typ,pdf_path.name,src))
        pages.sort(key=lambda x:x["pdf_page_index"])
        total=time.time()-tpdf
        timings={"manifest_texte":round(t_manifest,2),"rendu_images":round(t_render,2),"route_qwen":round(t_route,2),"extraction_qwen":round(t_extract,2),"retry_hd":retries,"total":round(total,2)}
        doc=build_document(pdf_path,pages,timings,(tok_in,tok_out)); doc["document"]["client_dossier"]=cid
        out=json_path_for_pdf(pdf_path); out.write_text(json.dumps(doc,ensure_ascii=False,indent=2,default=str),encoding="utf-8")
        index_records.append({"client":cid,"fichier":pdf_path.name,"json":str(out),"timings_s":timings,"tokens":tok_in+tok_out})
        log(prefix+f"OK {pdf_path.name} total={total:.1f}s extract={t_extract:.1f}s route={t_route:.1f}s")
        del rendered,by_idx; gc.collect()
    except Exception as e:
        log(prefix+"ERREUR "+pdf_path.name+" — "+type(e).__name__+": "+str(e))
        with open(LOG_PATH,"a",encoding="utf-8") as f: f.write(traceback.format_exc()+"\n")

with open(INDEX_PATH,"w",encoding="utf-8") as f:
    for r in index_records: f.write(json.dumps(r,ensure_ascii=False,default=str)+"\n")
print("✅ Extraction V24 terminée")


In [ ]:
# CELLULE 12 — MOTEUR DE CONTROLES DETERMINISTES (DZD)
# Les valeurs observées ne sont jamais remplacées par les valeurs calculées.
FORMULAS = [
 ("ACTIF","immobilisations_corporelles",["montant_brut","amortissements_provisions_pertes","net_n"],["terrains","batiments","autres_immobilisations_corporelles"]),
 ("ACTIF","immobilisations_financieres",["montant_brut","amortissements_provisions_pertes","net_n"],["titres_mis_en_equivalence","autres_participations_creances","autres_titres_immobilises","prets_actifs_financiers_non_courants","impots_differes_actif"]),
 ("ACTIF","creances_emplois_assimiles",["montant_brut","amortissements_provisions_pertes","net_n"],["clients","autres_debiteurs","impots_assimiles_actif","autres_creances_assimiles"]),
 ("ACTIF","disponibilites_assimiles",["montant_brut","amortissements_provisions_pertes","net_n"],["placements_financiers_courants","tresorerie_actif"]),
 ("ACTIF","total_actif_non_courant",["montant_brut","amortissements_provisions_pertes","net_n"],["ecarts_acquisition_goodwill","immobilisations_incorporelles","terrains","batiments","autres_immobilisations_corporelles","immobilisations_en_concession","immobilisations_en_cours","titres_mis_en_equivalence","autres_participations_creances","autres_titres_immobilises","prets_actifs_financiers_non_courants","impots_differes_actif"]),
 ("ACTIF","total_actif_courant",["montant_brut","amortissements_provisions_pertes","net_n"],["stocks_encours","clients","autres_debiteurs","impots_assimiles_actif","autres_creances_assimiles","placements_financiers_courants","tresorerie_actif"]),
 ("ACTIF","total_general_actif",["montant_brut","amortissements_provisions_pertes","net_n"],["total_actif_non_courant","total_actif_courant"]),
 ("PASSIF","total_capitaux_propres",["n","n1"],["capital_emis","primes_reserves","ecart_reevaluation","ecart_equivalence","resultat_net_passif","report_a_nouveau","part_societe_consolidante","part_minoritaires"],{"capital_non_appele":-1}),
 ("PASSIF","total_passifs_non_courants",["n","n1"],["emprunts_dettes_financieres","impots_differes_provisionnes","autres_dettes_non_courantes","provisions_produits_avance"]),
 ("PASSIF","total_passifs_courants",["n","n1"],["fournisseurs_rattaches","impots_passif","autres_dettes","tresorerie_passif"]),
 ("PASSIF","total_general_passif",["n","n1"],["total_capitaux_propres","total_passifs_non_courants","total_passifs_courants"]),
]
TCR_FORMULAS = {
 "chiffre_affaires_net":["ventes_marchandises","produits_fabriques","prestations_services","ventes_travaux","produits_annexes","rabais_remises_ristournes_accordes"],
 "production_exercice":["chiffre_affaires_net","production_stockee_destockee","production_immobilisee","subvention_exploitation"],
 "consommations_exercice":["achats_marchandises_vendues","matieres_premieres","autres_approvisionnements","variation_stocks","achats_etudes_prestations","autres_consommations","rabais_remises_ristournes_obtenus_achats","sous_traitance_generale","locations","entretien_reparations","primes_assurances","personnel_exterieur","remuneration_intermediaires","publicite","deplacements_missions","autres_services","rabais_remises_ristournes_obtenus_services"],
 "valeur_ajoutee_exploitation":["production_exercice","consommations_exercice"],
 "excedent_brut_exploitation":["valeur_ajoutee_exploitation","charges_personnel","impots_taxes_assimiles"],
 "resultat_operationnel":["excedent_brut_exploitation","autres_produits_operationnels","autres_charges_operationnelles","dotations_amortissements","provisions","pertes_valeur","reprises_pertes_valeur_provisions"],
 "resultat_financier":["produits_financiers","charges_financieres"],
 "resultat_ordinaire":["resultat_operationnel","resultat_financier"],
 "resultat_extraordinaire":["elements_extraordinaires_produits","elements_extraordinaires_charges"],
 "resultat_net_exercice":["resultat_ordinaire","resultat_extraordinaire","impots_exigibles_resultats","impots_differes_resultats"],
}

def index_doc(doc):
    idx={}
    for p in doc.get("pages",[]):
        for table,rows in (p.get("donnees") or {}).items():
            if not isinstance(rows,dict): continue
            for rc,vals in rows.items():
                if isinstance(vals,dict): idx.setdefault((table,rc),{}).update({k:v for k,v in vals.items() if v is not None})
    return idx

def tcr_net(vals, period="n"):
    d=vals.get(period+"_debit"); c=vals.get(period+"_credit")
    if d is None and c is None: return None
    return float(c or 0)-float(d or 0)

def add_control(out,kind,table,row,col,observed,calc,source=None):
    if observed is None or calc is None: return
    gap=float(observed)-float(calc); out.append({"kind":kind,"table_id":table,"row_id":row,"column_id":col,
        "observed_dzd":observed,"calculated_dzd":calc,"difference_dzd":round(gap,2),"level":ecart_level(gap),"source":source})

def run_document_controls(doc):
    idx=index_doc(doc); out=[]
    for formula in FORMULAS:
        table,target,cols,components=formula[:4]; overrides=formula[4] if len(formula)>4 else {}
        for col in cols:
            observed=(idx.get((table,target)) or {}).get(col)
            if observed is None: continue
            vals=[]; missing=False
            for rc in components:
                v=(idx.get((table,rc)) or {}).get(col)
                if v is None: missing=True; break
                vals.append(float(v)*float(overrides.get(rc,1)))
            # capital non appelé est un composant soustractif même s'il n'est pas dans la liste positive.
            for rc,mul in overrides.items():
                if rc in components: continue
                v=(idx.get((table,rc)) or {}).get(col)
                if v is None: missing=True; break
                vals.append(float(v)*float(mul))
            if not missing: add_control(out,"FORMULA_SCHEMA",table,target,col,observed,sum(vals))
    # ACTIF: brut - amort = net N ligne par ligne.
    for (table,rc),vals in idx.items():
        if table=="ACTIF" and vals.get("net_n") is not None and vals.get("montant_brut") is not None and vals.get("amortissements_provisions_pertes") is not None:
            add_control(out,"FORMULA_LINE","ACTIF",rc,"net_n",vals["net_n"],float(vals["montant_brut"])-float(vals["amortissements_provisions_pertes"]))
    # TCR: contrôle des soldes calculés sur N et N-1; les 4 cellules brutes restent intactes.
    for target,components in TCR_FORMULAS.items():
        for period in ("n","n1"):
            obs=tcr_net(idx.get(("TCR",target),{}),period)
            if obs is None: continue
            parts=[]; missing=False
            for rc in components:
                v=tcr_net(idx.get(("TCR",rc),{}),period)
                if v is None: missing=True; break
                parts.append(v)
            if not missing: add_control(out,"TCR_NET_SCHEMA","TCR",target,period+"_net",obs,sum(parts))
    # Contrôles croisés majeurs.
    a=(idx.get(("ACTIF","total_general_actif")) or {}).get("net_n"); p=(idx.get(("PASSIF","total_general_passif")) or {}).get("n")
    if a is not None and p is not None: add_control(out,"CROSS","BILAN","actif_passif","N",a,p)
    tn=tcr_net(idx.get(("TCR","resultat_net_exercice"),{}),"n"); pn=(idx.get(("PASSIF","resultat_net_passif")) or {}).get("n")
    if tn is not None and pn is not None: add_control(out,"CROSS","RESULTAT","tcr_passif","N",tn,pn)
    doc.setdefault("controles",{})["schema_deterministe"]={"seuils_dzd":{"LOW":ALERT_MIN_DZD,"MEDIUM":ALERT_MEDIUM_DZD,"HIGH":ALERT_HIGH_DZD},
        "details":out,"alerts":[x for x in out if x["level"]!="OK"],"nb_alerts":sum(x["level"]!="OK" for x in out)}
    return doc

# Enrichit les JSON V24 déjà extraits sans refaire Qwen.
for jf in JSON_DIR.rglob("*.json"):
    try:
        d=json.loads(jf.read_text(encoding="utf-8")); run_document_controls(d); jf.write_text(json.dumps(d,ensure_ascii=False,indent=2,default=str),encoding="utf-8")
    except Exception as e: log("CONTROLE JSON "+str(jf)+" — "+str(e))
print("✅ Contrôles déterministes ajoutés aux JSON")


In [ ]:
# CELLULE 13 — MAPPING EXCEL ANALYSTE V20 (VERIFIE SUR LE MODELE)
CFG_V20 = {'ACTIF_GROUPES': {'N': ['P', 'Q', 'R'], 'N-1': ['M', 'N', 'O'], 'N-2': ['J', 'K', 'L'], 'N-3': ['G', 'H', 'I'], 'N-4': ['D', 'E', 'F']},
 'PASSIF_COLS': {'N': 'H', 'N-1': 'G', 'N-2': 'F', 'N-3': 'E', 'N-4': 'D'},
 'TCR_PAIRES': {'N': ['M', 'N'], 'N-1': ['K', 'L'], 'N-2': ['I', 'J'], 'N-3': ['G', 'H'], 'N-4': ['E', 'F']},
 'ACTIF_ROWS': {'ecarts_acquisition_goodwill': 10,
                'immobilisations_incorporelles': 11,
                'immobilisations_corporelles': 12,
                'terrains': 13,
                'batiments': 14,
                'autres_immobilisations_corporelles': 15,
                'immobilisations_en_concession': 16,
                'immobilisations_en_cours': 17,
                'immobilisations_financieres': 18,
                'titres_mis_en_equivalence': 19,
                'autres_participations_creances': 20,
                'autres_titres_immobilises': 21,
                'prets_actifs_financiers_non_courants': 22,
                'impots_differes_actif': 23,
                'total_actif_non_courant': 24,
                'stocks_encours': 26,
                'creances_emplois_assimiles': 27,
                'clients': 28,
                'autres_debiteurs': 29,
                'impots_assimiles_actif': 30,
                'autres_creances_assimiles': 31,
                'disponibilites_assimiles': 32,
                'placements_financiers_courants': 33,
                'tresorerie_actif': 34,
                'total_actif_courant': 35,
                'total_general_actif': 36},
 'PASSIF_ROWS': {'capital_emis': 6,
                 'capital_non_appele': 7,
                 'primes_reserves': 8,
                 'ecart_reevaluation': 9,
                 'ecart_equivalence': 10,
                 'resultat_net_passif': 11,
                 'report_a_nouveau': 12,
                 'part_societe_consolidante': 13,
                 'part_minoritaires': 14,
                 'total_capitaux_propres': 15,
                 'emprunts_dettes_financieres': 17,
                 'impots_differes_provisionnes': 18,
                 'autres_dettes_non_courantes': 19,
                 'provisions_produits_avance': 20,
                 'total_passifs_non_courants': 21,
                 'fournisseurs_rattaches': 23,
                 'impots_passif': 24,
                 'autres_dettes': 25,
                 'tresorerie_passif': 26,
                 'total_passifs_courants': 27,
                 'total_general_passif': 28},
 'TCR_ROWS': {'ventes_marchandises': 6,
              'produits_fabriques': 7,
              'prestations_services': 8,
              'ventes_travaux': 9,
              'produits_annexes': 10,
              'rabais_remises_ristournes_accordes': 11,
              'chiffre_affaires_net': 12,
              'production_stockee_destockee': 13,
              'production_immobilisee': 14,
              'subvention_exploitation': 15,
              'production_exercice': 16,
              'achats_marchandises_vendues': 17,
              'matieres_premieres': 18,
              'autres_approvisionnements': 19,
              'variation_stocks': 20,
              'achats_etudes_prestations': 21,
              'autres_consommations': 22,
              'rabais_remises_ristournes_obtenus_achats': 23,
              'sous_traitance_generale': 24,
              'locations': 25,
              'entretien_reparations': 26,
              'primes_assurances': 27,
              'personnel_exterieur': 28,
              'remuneration_intermediaires': 29,
              'publicite': 30,
              'deplacements_missions': 31,
              'autres_services': 32,
              'rabais_remises_ristournes_obtenus_services': 33,
              'consommations_exercice': 34,
              'valeur_ajoutee_exploitation': 35,
              'charges_personnel': 36,
              'impots_taxes_assimiles': 37,
              'excedent_brut_exploitation': 38,
              'autres_produits_operationnels': 39,
              'autres_charges_operationnelles': 40,
              'dotations_amortissements': 41,
              'provisions': 42,
              'pertes_valeur': 43,
              'reprises_pertes_valeur_provisions': 44,
              'resultat_operationnel': 45,
              'produits_financiers': 46,
              'charges_financieres': 47,
              'resultat_financier': 48,
              'resultat_ordinaire': 49,
              'elements_extraordinaires_produits': 50,
              'elements_extraordinaires_charges': 51,
              'resultat_extraordinaire': 52,
              'impots_exigibles_resultats': 53,
              'impots_differes_resultats': 54,
              'resultat_net_exercice': 55},
 'ACTIF_FORMULES': [9, 12, 18, 24, 25, 27, 32, 35, 36],
 'PASSIF_FORMULES': [5, 15, 16, 21, 22, 27, 28],
 'TCR_FORMULES': [12, 16, 34, 35, 38, 45, 48, 49, 52, 55]}
FEUILLE_ACTIF="Saisie actif"; FEUILLE_PASSIF="Saisie passif"; FEUILLE_TCR="Saisie TCR"
RANKS=["N","N-1","N-2","N-3","N-4"]

# Résolution modèle / recalcul dans Domino, avec fallback /mnt/data pour test local.
def resolve_model():
    cands=[Path("/mnt/Risk/Model/MODEL ETAT FINANCIER - ANALYSTE COPILOTE V20.xlsx"),Path("/mnt/data/MODEL ETAT FINANCIER - ANALYSTE COPILOTE V20.xlsx")]
    for p in cands:
        if p.exists(): return p
    more=list(Path("/mnt/Risk/Model").glob("*ANALYSTE*V20*.xlsx")) if Path("/mnt/Risk/Model").exists() else []
    return more[0] if more else cands[0]

def resolve_recalc():
    for p in [Path("/mnt/Risk/Model/recalc_liasse.py"),Path("/mnt/data/recalc_liasse.py")]:
        if p.exists(): return p
    return Path("/mnt/Risk/Model/recalc_liasse.py")

MODEL_XLSX=resolve_model(); RECALC_TOOL=resolve_recalc()
print("Modèle Excel:",MODEL_XLSX)
print("Recalc tool:",RECALC_TOOL,"| soffice:",shutil.which("soffice"))


In [ ]:
# CELLULE 14 — CONSTRUCTION EXCEL MULTI-EXERCICES + SOURCES + IDENTITE
def is_formula(cell): return isinstance(cell.value,str) and cell.value.startswith("=")
def kdzd(v): return None if v is None else float(v)/1000.0

def table_rows(doc,table):
    out={}; pages=[]
    for p in doc.get("pages",[]):
        if table not in (p.get("classification") or {}).get("types",[]): continue
        rows=(p.get("donnees") or {}).get(table)
        if isinstance(rows,dict):
            for rc,vals in rows.items(): out.setdefault(rc,{}).update({k:v for k,v in vals.items() if v is not None})
            pages.append(p.get("numero_page_scannee"))
    return out,[x for x in pages if x]

def source_text(doc,table,period,year):
    _,pages=table_rows(doc,table); f=(doc.get("document") or {}).get("fichier_source") or "?"
    pp="p."+",".join(map(str,pages)) if pages else "page ?"
    return f"{year} — {f} — {pp}" + (" — colonne N-1" if period=="N1" else "")

def doc_year(doc):
    y=(doc.get("identite") or {}).get("annee")
    if y: return int(y)
    return parse_year((doc.get("identite") or {}).get("exercice_au")) or parse_year((doc.get("document") or {}).get("fichier_source"))

def choose_candidates(docs):
    # Choix par (année, tableau), priorité au N de son propre PDF, puis au N-1 du PDF suivant.
    cands=defaultdict(list)
    for d in docs:
        y=doc_year(d)
        if not y: continue
        for table in ("ACTIF","PASSIF","TCR"):
            rows,_=table_rows(d,table)
            if rows: cands[(y,table)].append((0,d,"N"))
            # Fallback N-1 uniquement PASSIF/TCR : le formulaire ACTIF N-1 ne fournit que le Net.
            # On refuse de fabriquer Brut/Amort pour satisfaire les formules du modèle Excel.
            if table in {"PASSIF","TCR"} and rows and any(any(k.startswith("n1") for k in vals) for vals in rows.values()):
                cands[(y-1,table)].append((1,d,"N1"))
    return {k:sorted(v,key=lambda x:x[0])[0] for k,v in cands.items()}

def majority_identity(docs):
    def maj(key):
        vals=[(d.get("identite") or {}).get(key) for d in docs]; vals=[str(x) for x in vals if x not in (None,"")]
        return Counter(vals).most_common(1)[0][0] if vals else ""
    return {"entreprise":maj("entreprise"),"nif_15":maj("nif_15")}

def clear_writable_zones(wb):
    for rank in RANKS:
        ga=CFG_V20["ACTIF_GROUPES"][rank]; cp=CFG_V20["PASSIF_COLS"][rank]; gt=CFG_V20["TCR_PAIRES"][rank]
        for rw in CFG_V20["ACTIF_ROWS"].values():
            for c in ga:
                cell=wb[FEUILLE_ACTIF][f"{c}{rw}"]
                if not is_formula(cell): cell.value=None; cell.comment=None
        for rw in CFG_V20["PASSIF_ROWS"].values():
            cell=wb[FEUILLE_PASSIF][f"{cp}{rw}"]
            if not is_formula(cell): cell.value=None; cell.comment=None
        for rw in CFG_V20["TCR_ROWS"].values():
            for c in gt:
                cell=wb[FEUILLE_TCR][f"{c}{rw}"]
                if not is_formula(cell): cell.value=None; cell.comment=None

def collect_formula_control(controls,cell,observed,meta):
    if observed is None: return
    rec=dict(meta); rec.update({"sheet":cell.parent.title,"cell":cell.coordinate,"observed_kdzd":kdzd(observed)})
    controls.append(rec)

def write_one_table(wb,table,rank,doc,period,year,controls):
    rows,_=table_rows(doc,table); written=0
    src=source_text(doc,table,period,year)
    if table=="ACTIF":
        cols=CFG_V20["ACTIF_GROUPES"][rank]
        # N -> brut/amort/net. N1 -> uniquement Net N-1 : ne jamais fabriquer Brut/Amort pour alimenter une formule.
        mapping=[("montant_brut",cols[0]),("amortissements_provisions_pertes",cols[1]),("net_n",cols[2])] if period=="N" else [("net_n1",cols[2])]
        for rc,rw in CFG_V20["ACTIF_ROWS"].items():
            vals=rows.get(rc,{})
            for field,col in mapping:
                v=vals.get(field)
                if v is None: continue
                cell=wb[FEUILLE_ACTIF][f"{col}{rw}"]
                if is_formula(cell): collect_formula_control(controls,cell,v,{"table":"ACTIF","row":rc,"column":field,"source":src,"year":year})
                else: cell.value=kdzd(v); cell.number_format='#,##0'; written+=1
    elif table=="PASSIF":
        col=CFG_V20["PASSIF_COLS"][rank]; field="n" if period=="N" else "n1"
        for rc,rw in CFG_V20["PASSIF_ROWS"].items():
            v=rows.get(rc,{}).get(field)
            if v is None: continue
            cell=wb[FEUILLE_PASSIF][f"{col}{rw}"]
            if is_formula(cell): collect_formula_control(controls,cell,v,{"table":"PASSIF","row":rc,"column":field,"source":src,"year":year})
            else: cell.value=kdzd(v); cell.number_format='#,##0'; written+=1
    elif table=="TCR":
        dcol,ccol=CFG_V20["TCR_PAIRES"][rank]; df="n_debit" if period=="N" else "n1_debit"; cf="n_credit" if period=="N" else "n1_credit"
        for rc,rw in CFG_V20["TCR_ROWS"].items():
            vals=rows.get(rc,{})
            for field,col in ((df,dcol),(cf,ccol)):
                v=vals.get(field)
                if v is None: continue
                cell=wb[FEUILLE_TCR][f"{col}{rw}"]
                # Comparaison Débit et Crédit séparément, jamais seulement un net.
                if is_formula(cell): collect_formula_control(controls,cell,v,{"table":"TCR","row":rc,"column":field,"source":src,"year":year})
                else: cell.value=kdzd(v); cell.number_format='#,##0'; written+=1
    return written,src

def hide_missing_years(wb,year_to_rank,present_years):
    present_ranks={year_to_rank[y] for y in present_years if y in year_to_rank}
    for rank in RANKS:
        hide=rank not in present_ranks
        for c in CFG_V20["ACTIF_GROUPES"][rank]: wb[FEUILLE_ACTIF].column_dimensions[c].hidden=hide
        wb[FEUILLE_PASSIF].column_dimensions[CFG_V20["PASSIF_COLS"][rank]].hidden=hide
        for c in CFG_V20["TCR_PAIRES"][rank]: wb[FEUILLE_TCR].column_dimensions[c].hidden=hide

def write_headers_sources(wb,identity,year_to_rank,sources):
    # raison sociale + NIF au-dessus de l'exercice N pour les 3 états
    for ws,col in [(wb[FEUILLE_ACTIF],CFG_V20["ACTIF_GROUPES"]["N"][0]),(wb[FEUILLE_PASSIF],CFG_V20["PASSIF_COLS"]["N"]),(wb[FEUILLE_TCR],CFG_V20["TCR_PAIRES"]["N"][0])]:
        ws[f"{col}1"]=identity.get("entreprise") or ""; ws[f"{col}2"].number_format='@'; ws[f"{col}2"]=identity.get("nif_15") or ""
    # source au-dessous de chaque exercice, séparée par état
    for year,rank in year_to_rank.items():
        for table,ws,col in [
            ("ACTIF",wb[FEUILLE_ACTIF],CFG_V20["ACTIF_GROUPES"][rank][0]),
            ("PASSIF",wb[FEUILLE_PASSIF],CFG_V20["PASSIF_COLS"][rank]),
            ("TCR",wb[FEUILLE_TCR],CFG_V20["TCR_PAIRES"][rank][0])]:
            cell=ws[f"{col}3"]; cell.value=sources.get((year,table),""); cell.alignment=Alignment(wrap_text=True,vertical="top"); cell.font=Font(size=8,italic=True)
    for ws in (wb[FEUILLE_ACTIF],wb[FEUILLE_PASSIF],wb[FEUILLE_TCR]): ws.row_dimensions[3].height=max(ws.row_dimensions[3].height or 15,30)


In [ ]:
# CELLULE 15 — RECALCUL + COMPARAISON PDF/QWEN vs FORMULE EXCEL (LOW/MEDIUM/HIGH)
FILL_LOW=PatternFill("solid",start_color="FFF2CC")
FILL_MEDIUM=PatternFill("solid",start_color="F8CBAD")
FILL_HIGH=PatternFill("solid",start_color="FFC7CE")
COLOR_LOW="C9A227"; COLOR_MEDIUM="ED7D31"; COLOR_HIGH="FF0000"

def border_alert(cell,color,style="medium"):
    s=Side(style=style,color=color); cell.border=Border(left=s,right=s,top=s,bottom=s)

def recalc_excel(path,timeout=60):
    """Recalcule les formules via LibreOffice en conversion xlsx->xlsx.
    Cette voie est nettement plus rapide que la macro historique. Fallback vers recalc_liasse.py si nécessaire.
    """
    path=Path(path)
    soffice=shutil.which("soffice") or shutil.which("libreoffice")
    if soffice:
        try:
            with tempfile.TemporaryDirectory(prefix="yarisk_lo_") as td:
                td=Path(td); outdir=td/"out"; outdir.mkdir(); profile=td/"profile"
                cmd=[soffice,"--headless","--norestore",f"-env:UserInstallation={profile.as_uri()}",
                     "--convert-to","xlsx","--outdir",str(outdir),str(path.resolve())]
                r=subprocess.run(cmd,capture_output=True,text=True,timeout=timeout)
                converted=outdir/path.name
                if r.returncode==0 and converted.exists():
                    shutil.copy2(converted,path)
                    return {"ok":True,"code":0,"message":"OK LibreOffice fast convert","stdout":(r.stdout or "")[-500:]}
        except Exception as e:
            fast_error=type(e).__name__+": "+str(e)
        else:
            fast_error="conversion LibreOffice sans fichier de sortie"
    else:
        fast_error="soffice indisponible"
    if RECALC_TOOL.exists() and soffice:
        try:
            r=subprocess.run([sys.executable,str(RECALC_TOOL),str(path),str(min(timeout,90))],capture_output=True,text=True,timeout=min(timeout,90)+30)
            return {"ok":r.returncode==0,"code":r.returncode,"message":"OK fallback recalc_liasse.py" if r.returncode==0 else (r.stderr or r.stdout)[-1000:],"fast_error":fast_error}
        except Exception as e:
            return {"ok":False,"message":fast_error+" | fallback "+type(e).__name__+": "+str(e)}
    return {"ok":False,"message":fast_error}

def scan_bad_formulas(wb):
    out=[]
    for ws in wb.worksheets:
        for row in ws.iter_rows():
            for c in row:
                if isinstance(c.value,str) and c.value.startswith("=") and "#REF!" in c.value.upper(): out.append(f"{ws.title}!{c.coordinate} = {c.value}")
    return out

def compare_formula_cells(path,controls):
    wbf=load_workbook(path); wbv=load_workbook(path,data_only=True); results=[]
    for ctrl in controls:
        observed=ctrl.get("observed_kdzd"); calc=wbv[ctrl["sheet"]][ctrl["cell"]].value
        rec=dict(ctrl)
        if not isinstance(calc,(int,float)) or observed is None:
            rec.update({"status":"NON_CALCULE","level":"NON_CALCULE","calculated_kdzd":calc}); results.append(rec); continue
        diff_kdzd=float(calc)-float(observed); diff_dzd=diff_kdzd*1000.0; level=ecart_level(diff_dzd)
        rec.update({"calculated_kdzd":float(calc),"difference_kdzd":round(diff_kdzd,6),"difference_dzd":round(diff_dzd,2),"level":level,"status":"OK" if level=="OK" else "ALERTE"})
        results.append(rec)
        if level=="OK": continue
        cell=wbf[ctrl["sheet"]][ctrl["cell"]]
        pct=None
        if abs(float(observed))>1e-12: pct=100.0*diff_kdzd/abs(float(observed))
        txt=("CONTROLE YA-RISK V24 — PDF/QWEN vs FORMULE\n\n"
             f"Tableau : {ctrl.get('table')}\nPoste : {ctrl.get('row')}\nCase : {ctrl.get('column')}\nExercice : {ctrl.get('year')}\n"
             f"Valeur PDF/Qwen : {observed:,.3f} KDZD\nValeur formule : {float(calc):,.3f} KDZD\n"
             f"Ecart : {diff_dzd:+,.0f} DZD\nNiveau : {level}\n" + (f"Ecart relatif : {pct:+.4f}%\n" if pct is not None else "") +
             f"Source : {ctrl.get('source','')}")
        cell.comment=Comment(txt,"Ya-Risk")
        if level=="LOW": cell.fill=FILL_LOW; border_alert(cell,COLOR_LOW,"medium")
        elif level=="MEDIUM": cell.fill=FILL_MEDIUM; border_alert(cell,COLOR_MEDIUM,"medium")
        else: cell.fill=FILL_HIGH; border_alert(cell,COLOR_HIGH,"thick")
    wbf.save(path); return results

def add_control_sheet(path,client,identity,years,sources,formula_results,cross_doc_alerts,bad_formulas,recalc_info):
    wb=load_workbook(path)
    name="0.Controles YaRisk V24"
    if name in wb.sheetnames: del wb[name]
    ws=wb.create_sheet(name,0); ws["A1"]="YA-RISK — CONTROLES PRE-CERTIFICATION V24"; ws["A1"].font=Font(bold=True,size=14)
    rows=[["Client",client],["Raison sociale",identity.get("entreprise")],["NIF",identity.get("nif_15")],["Exercices",", ".join(map(str,years))],
          ["Seuil LOW à partir de (DZD)",ALERT_MIN_DZD],["MEDIUM à partir de (DZD)",ALERT_MEDIUM_DZD],["HIGH à partir de (DZD)",ALERT_HIGH_DZD],["Recalcul LibreOffice",recalc_info.get("message")]]
    for r in rows: ws.append(r)
    ws.append([]); ws.append(["ECARTS FORMULES", "Niveau","Exercice","Tableau","Poste","Case","PDF/Qwen KDZD","Formule KDZD","Ecart DZD","Source"])
    for x in formula_results:
        if x.get("level") in ("LOW","MEDIUM","HIGH","NON_CALCULE"):
            ws.append([x.get("status"),x.get("level"),x.get("year"),x.get("table"),x.get("row"),x.get("column"),x.get("observed_kdzd"),x.get("calculated_kdzd"),x.get("difference_dzd"),x.get("source")])
    ws.append([]); ws.append(["ECARTS INTER-DOCUMENTS","Niveau","Année","Field","Valeur 1 DZD","Valeur 2 DZD","Ecart DZD","Source 1","Source 2"])
    for x in cross_doc_alerts:
        ws.append(["ALERTE",x.get("level"),x.get("year"),x.get("field_key"),x.get("value1"),x.get("value2"),x.get("difference_dzd"),x.get("source1"),x.get("source2")])
    ws.append([]); ws.append(["ANOMALIES MODELE EXCEL"])
    for x in bad_formulas: ws.append([x])
    ws.freeze_panes="A2"
    for c,w in {"A":28,"B":14,"C":12,"D":16,"E":32,"F":22,"G":18,"H":18,"I":18,"J":52}.items(): ws.column_dimensions[c].width=w
    for row in ws.iter_rows():
        for c in row: c.alignment=Alignment(wrap_text=True,vertical="top")
    wb.save(path)


In [ ]:
# CELLULE 16 — CONTROLE INTER-DOCUMENTS + POST-TRAITEMENT CLIENT
def canonical_year_observations(docs):
    # clé = (année, table, row, côté/colonne). On conserve toutes les sources pour comparer N vs N-1.
    bag=defaultdict(list)
    for d in docs:
        y=doc_year(d)
        if not y: continue
        f=(d.get("document") or {}).get("fichier_source") or "?"
        for p in d.get("pages",[]):
            page=p.get("numero_page_scannee")
            for table,rows in (p.get("donnees") or {}).items():
                if table not in ("ACTIF","PASSIF","TCR") or not isinstance(rows,dict): continue
                for rc,vals in rows.items():
                    for col,v in vals.items():
                        if v is None: continue
                        if table=="ACTIF":
                            if col=="net_n1": yy=y-1; keycol="net"
                            elif col in ("montant_brut","amortissements_provisions_pertes","net_n"): yy=y; keycol=col.replace("net_n","net")
                            else: continue
                        elif table=="PASSIF": yy=y if col=="n" else y-1 if col=="n1" else None; keycol="value"
                        else:
                            if col.startswith("n1_"): yy=y-1; keycol=col[3:]
                            elif col.startswith("n_"): yy=y; keycol=col[2:]
                            else: continue
                        if yy is not None: bag[(yy,table,rc,keycol)].append({"value":float(v),"source":f"{f} — p.{page} — {col}"})
    return bag

def cross_document_alerts(docs):
    out=[]
    for (year,table,rc,col),obs in canonical_year_observations(docs).items():
        if len(obs)<2: continue
        # compare chaque observation à la première; aucune écrasée.
        base=obs[0]
        for other in obs[1:]:
            diff=other["value"]-base["value"]; level=ecart_level(diff)
            if level!="OK": out.append({"year":year,"field_key":f"{table}.{rc}.{col}","value1":base["value"],"value2":other["value"],"difference_dzd":round(diff,2),"level":level,"source1":base["source"],"source2":other["source"]})
    return out

def build_client_excel(client,docs):
    if not MODEL_XLSX.exists(): raise FileNotFoundError("Modèle Excel introuvable: "+str(MODEL_XLSX))
    years_raw=set()
    for d in docs:
        y=doc_year(d)
        if y: years_raw.update([y,y-1])
    # Un exercice N-1 n'est affiché que si au moins PASSIF/TCR/ACTIF contient réellement une valeur N-1.
    candidates=choose_candidates(docs); years=sorted({y for (y,t) in candidates},reverse=True)
    if not years: raise ValueError("Aucun exercice détecté pour "+client)
    latest=max(years); year_to_rank={latest-i:RANKS[i] for i in range(5)}; years_excel=[y for y in years if y in year_to_rank]

    out=XLSX_DIR/("PRECERT_V24_"+re.sub(r"[^A-Za-z0-9_.-]+","_",client)+".xlsx")
    shutil.copy2(MODEL_XLSX,out); wb=load_workbook(out); clear_writable_zones(wb)
    bad=scan_bad_formulas(wb); controls=[]; sources={}; written=0
    for (year,table),(prio,doc,period) in candidates.items():
        if year not in year_to_rank: continue
        n,src=write_one_table(wb,table,year_to_rank[year],doc,period,year,controls); written+=n; sources[(year,table)]=src
    ident=majority_identity(docs); write_headers_sources(wb,ident,year_to_rank,sources); hide_missing_years(wb,year_to_rank,years_excel)
    try:
        wb.calculation.fullCalcOnLoad=True; wb.calculation.forceFullCalc=True; wb.calculation.calcMode="auto"
    except Exception: pass
    wb.save(out)

    recalc1=recalc_excel(out)
    formula_results=compare_formula_cells(out,controls) if recalc1.get("ok") else [dict(x,status="NON_CALCULE",level="NON_CALCULE") for x in controls]
    cross=cross_document_alerts(docs)
    add_control_sheet(out,client,ident,years_excel,sources,formula_results,cross,bad,recalc1)
    recalc2=recalc_excel(out)
    return out,{"client":client,"identity":ident,"years":years_excel,"written":written,"sources":{f"{y}.{t}":s for (y,t),s in sources.items()},
                "formula_controls":formula_results,"cross_document_alerts":cross,"model_formula_warnings":bad,"recalc_initial":recalc1,"recalc_final":recalc2}

clients=defaultdict(list)
for jf in JSON_DIR.rglob("*.json"):
    rel=jf.relative_to(JSON_DIR); client=rel.parts[0] if len(rel.parts)>1 else "_RACINE_"
    try: clients[client].append(json.loads(jf.read_text(encoding="utf-8")))
    except Exception as e: log("JSON illisible "+str(jf)+" — "+str(e))

log(f"V24 post-traitement — {len(clients)} client(s)")
for client,docs in sorted(clients.items()):
    try:
        out,report=build_client_excel(client,docs)
        bundle={"schema_version":SCHEMA_VERSION,"schema_registry_version":SCHEMA_REGISTRY_VERSION,"workflow":{"statut":"A_CERTIFIER","certifie":False,"version_pipeline":"V24_SCHEMA_FIRST"},
                "dossier":{"reference":client},"documents":docs,"excel_pre_certification":str(out),"rapport_excel":report}
        jp=CONTROL_DIR/(re.sub(r"[^A-Za-z0-9_.-]+","_",client)+".avant_certification.json")
        jp.write_text(json.dumps(bundle,ensure_ascii=False,indent=2,default=str),encoding="utf-8")
        print("CLIENT",client,"| exercices",report["years"],"| écrits",report["written"],"| alertes formule",sum(x.get("level") in ("LOW","MEDIUM","HIGH") for x in report["formula_controls"]),"| Excel",out)
    except Exception as e:
        log("POST-TRAITEMENT "+client+" — "+type(e).__name__+": "+str(e)); traceback.print_exc()

print("✅ V24 terminé — Excel:",XLSX_DIR)
print("✅ JSON audit:",CONTROL_DIR)


In [ ]:
# CELLULE 17 — TESTS DE COHERENCE V24 (A EXECUTER)
_tests=[]
def check(name,cond,detail=""):
    ok=bool(cond); _tests.append(ok); print(("✅" if ok else "❌"),name,detail if not ok else ""); return ok

# Contrat unique
check("field_id uniques",len(FIELD_REGISTRY)==len(set(FIELD_REGISTRY)))
check("TCR 4 colonnes exactes",TABLE_SCHEMAS["TCR"]["cols"]==["n_debit","n_credit","n1_debit","n1_credit"])
check("TCR P1/P2 couvrent chaque ligne une fois",set(TCR_P1_ROWS).isdisjoint(TCR_P2_ROWS) and set(TCR_P1_ROWS+TCR_P2_ROWS)==set(TABLE_SCHEMAS["TCR"]["postes"]))

# Cas de confusion critiques
check("A09 n'est jamais TCR",route_from_text("9/ Tableau de détermination du résultat fiscal I. Résultat net de l'exercice II. Réintégrations III. Déductions Résultat fiscal")=="A09")
check("Annexe 3/4 n'est jamais TCR2",route_from_text("3/ Charges de personnel, impôts, taxes 4/ Autres charges et produits opérationnels Autres produits opérationnels")=="A03_A04")
check("TCR P2 sans entête reconnu",route_from_text("Autres produits opérationnels V-Résultat opérationnel VI-Résultat financier VII-Résultat ordinaire VIII-Résultat extraordinaire IX - RESULTAT NET DE L’EXERCICE")=="TCR_P2")
check("PASSIF structure officielle",route_from_text("BILAN (PASSIF) CAPITAUX PROPRES PASSIFS NON-COURANTS PASSIFS COURANTS TOTAL PASSIF (I+II+III)")=="PASSIF")

# Seuils d'écart demandés
check("999 DZD = OK",ecart_level(999)=="OK")
check("1000 DZD = LOW",ecart_level(1000)=="LOW")
check("10000 DZD = MEDIUM",ecart_level(10000)=="MEDIUM")
check("100000 DZD = HIGH",ecart_level(100000)=="HIGH")

# Validation optionnelle du formulaire de référence s'il est présent dans le runtime.
ref=Path("/mnt/data/GN2_administration_uniquement.pdf")
if ref.exists():
    expected=["ACTIF","PASSIF","TCR_P1","TCR_P2","A01_A02","A03_A04","A05_A06","A07_A08","A08_1_A08_2","A09","A10_A11","A12_A13"]
    got=[route_from_text(p["text"]) for p in pdf_manifest(ref)]
    check("12 pages formulaire G2 routées exactement",got==expected,"got="+repr(got))

# Vérification du modèle Excel sans le modifier.
if MODEL_XLSX.exists():
    wbtest=load_workbook(MODEL_XLSX,read_only=False,data_only=False)
    check("Feuilles Excel attendues",all(s in wbtest.sheetnames for s in [FEUILLE_ACTIF,FEUILLE_PASSIF,FEUILLE_TCR]))
    check("PASSIF total N est formule",is_formula(wbtest[FEUILLE_PASSIF]["H28"]))
    check("TCR résultat net N Débit est formule",is_formula(wbtest[FEUILLE_TCR]["M55"]))
    check("TCR résultat net N Crédit est formule",is_formula(wbtest[FEUILLE_TCR]["N55"]))

print(f"\nTESTS V24: {sum(_tests)}/{len(_tests)} réussis")
if not all(_tests): raise AssertionError("Au moins un test V24 a échoué — ne pas industrialiser avant correction")


## Sorties

- JSON V24 : `/mnt/Risk/bilans_out/GT/V24/json_bruts_v24/<CLIENT>/...json`
- Excel pré-certification : `/mnt/Risk/bilans_out/GT/V24/excel_pre_certification_v24/`
- JSON d'audit : `/mnt/Risk/bilans_out/GT/V24/json_avant_certification_v24/`

Le premier lancement V24 ré-extrait les PDF dans son propre répertoire. Les relances réutilisent automatiquement les JSON V24 déjà produits, sauf si `FORCE_REEXTRACT=True`.
